# Trabajo Práctico: Visualización de datos con la API Data360 del Banco Mundial
**Tema: la excepción argentina. Inflación y crecimiento en América Latina, 1960-2025**

Visualización de la Información · ITBA · 2026

**Integrantes:** Nombre Apellido · Nombre Apellido · Nombre Apellido

| Punto de la consigna | Qué hacemos | Dónde |
|---|---|---|
| 1. Explorar la API | Endpoints, bases de datos, indicadores y estructura de los datos | Sección 1 |
| 2. Visualizaciones | Tres gráficos interactivos en Altair sobre inflación y crecimiento en América Latina | Sección 2 |
| 3. Storytelling | Los números de la historia. La nota se publica como landing en GitHub Pages (`docs/index.html`) con los gráficos exportados acá | Sección 3 |

Todos los datos se descargan en vivo de la API Data360 (`https://data360api.worldbank.org`).

## 0. Configuración
Imports, la URL de la API y tres funciones para consultarla. Todo el notebook usa solo estas funciones.

In [1]:
from pathlib import Path

import altair as alt
import pandas as pd
import requests

API = "https://data360api.worldbank.org/data360"


def api_get(ruta, **params):
    """Consulta un endpoint GET de la API y devuelve el JSON."""
    respuesta = requests.get(f"{API}/{ruta}", params=params, timeout=60)
    respuesta.raise_for_status()
    return respuesta.json()


def api_post(ruta, cuerpo):
    """Consulta un endpoint POST de la API (búsqueda y metadatos) y devuelve el JSON."""
    respuesta = requests.post(f"{API}/{ruta}", json=cuerpo, timeout=60)
    respuesta.raise_for_status()
    return respuesta.json()


def obtener_datos(base, indicador, **filtros):
    """Descarga todas las observaciones de un indicador.
    /data devuelve como máximo 1000 filas por llamada, así que pedimos páginas con `skip` hasta completar `count`."""
    filas = []
    while True:
        pagina = api_get("data", DATABASE_ID=base, INDICATOR=indicador, skip=len(filas), **filtros)
        filas += pagina["value"]
        if len(filas) >= pagina["count"] or not pagina["value"]:
            return pd.DataFrame(filas)

Estilo común a todos los gráficos: tipografía, grises para ejes y textos, y un azul de acento, con colores validados para daltonismo y contraste. El tema también define el formato de números en español (`1.000,5`) y viaja dentro de cada gráfico exportado.

In [2]:
FONDO, TINTA, TINTA_SECUNDARIA, TINTA_TENUE = "#fcfcfb", "#0b0b0b", "#52514e", "#898781"
GRILLA, EJE = "#e1e0d9", "#c3c2b7"
AZUL = "#2a78d6"  # color de acento


@alt.theme.register("nota", enable=True)
def tema_nota():
    return {
        "config": {
            "background": FONDO,
            "font": "system-ui, -apple-system, 'Segoe UI', sans-serif",
            "view": {"stroke": None},
            "title": {"anchor": "start", "fontSize": 17, "fontWeight": 600, "color": TINTA,
                      "subtitleFontSize": 13, "subtitleColor": TINTA_SECUNDARIA, "subtitlePadding": 6, "offset": 16},
            "axis": {"labelColor": TINTA_TENUE, "labelFontSize": 11, "titleColor": TINTA_SECUNDARIA,
                     "titleFontSize": 12, "titleFontWeight": 500, "gridColor": GRILLA, "domainColor": EJE, "tickColor": EJE},
            "legend": {"labelColor": TINTA_SECUNDARIA, "labelFontSize": 12},
            "text": {"color": TINTA_SECUNDARIA, "fontSize": 12},
            "locale": {"number": {"decimal": ",", "thousands": ".", "grouping": [3], "currency": ["US$ ", ""]}},
        }
    }

## 1. Exploración de la API (punto 1)

### 1.1 ¿Qué ofrece la API?
La página [data360.worldbank.org/en/api](https://data360.worldbank.org/en/api) muestra un Swagger construido a partir de una especificación OpenAPI pública. La leemos para listar los endpoints:

In [3]:
spec = requests.get("https://raw.githubusercontent.com/worldbank/open-api-specs/main/Data360%20Open_API.json", timeout=60).json()
pd.DataFrame(
    [(metodo.upper(), ruta, operacion["summary"]) for ruta, operaciones in spec["paths"].items() for metodo, operacion in operaciones.items()],
    columns=["Método", "Endpoint", "Descripción"],
)

,Método,Endpoint,Descripción
0,GET,/data360/data,Data based on provided filters
1,GET,/data360/disaggregation,Disaggregations for a given indicator
2,GET,/data360/indicators,List of indicator IDs
3,POST,/data360/metadata,Metadata based on provided filters
4,POST,/data360/searchv2,Search function


| Endpoint | Para qué lo usamos |
|---|---|
| `POST /searchv2` | Buscador del catálogo: lista las bases (documentos `type eq 'dataset'`) y busca indicadores por palabra clave |
| `GET /indicators` | Códigos de todos los indicadores de una base |
| `POST /metadata` | Ficha de un indicador: nombre, definición, unidad, periodicidad, fuente |
| `GET /disaggregation` | Qué desagregaciones (sexo, edad, etc.) tiene un indicador |
| `GET /data` | Los datos en sí: una fila por observación |

Además, el portal usa `GET /portal/v1/hierarchy`, que no figura en el Swagger, para agrupar países por región y por nivel de ingreso. Lo usamos para separar países de agregados regionales.

### 1.2 ¿Qué bases de datos existen?
No hay un endpoint para listar bases, pero el buscador indexa cada base como un documento de tipo `dataset` con su ficha completa. Pedimos todas:

In [4]:
campos = ["database_id", "name", "sources", "topics", "indicator_count", "economies_count", "time_periods"]
respuesta = api_post("searchv2", {
    "search": "*", "filter": "type eq 'dataset'", "top": 1000,
    "select": ", ".join(f"series_description/{campo}" for campo in campos),
})

bases = pd.DataFrame([
    {
        "base": ficha["database_id"],
        "sigla": ficha["database_id"].split("_")[0],  # el código empieza con la sigla de la organización: WB_WDI → WB
        "nombre": ficha["name"],
        "fuente": ficha["sources"][0]["organization"] if ficha["sources"] else None,
        "tema": ", ".join(t["name"] for t in ficha["topics"] or [] if t["vocabulary"] == "WB Practice Groups" and t["parent_id"] is None),
        "indicadores": ficha["indicator_count"],
        "economías": ficha["economies_count"],
        "período": f"{ficha['time_periods'][0]['start']}-{ficha['time_periods'][0]['end']}",
    }
    for ficha in (documento["series_description"] for documento in respuesta["value"])
])
print(f"{len(bases)} bases de datos de {bases['sigla'].nunique()} organizaciones, con {bases['indicadores'].sum()} indicadores en total")
bases.sort_values("indicadores", ascending=False).head(15)

174 bases de datos de 52 organizaciones, con 9985 indicadores en total


,base,sigla,nombre,fuente,tema,indicadores,economías,período
56,WB_WDI,WB,World Development Indicators (WDI),World Bank (WB),"Planet, People, Prosperity, Infrastructure",1545,217,1960-2025
90,IMF_BOP,IMF,Balance of Payments (BOP) and International In...,International Monetary Fund (IMF),Prosperity,1146,198,1948-2024
20,WB_EDSTATS,WB,Education Statistics,World Bank (WB),"People, Prosperity",1072,222,1970-2023
109,WB_ES,WB,Enterprise Surveys,World Bank (WB),"Prosperity, Infrastructure, Digital",584,181,2006.0-2026.0
24,IMF_IFS,IMF,International Financial Statistics (IFS),International Monetary Fund (IMF),Prosperity,495,206,1940-2024
49,WB_GS,WB,Gender Statistics,World Bank (WB),"People, Prosperity, Infrastructure",363,217,1960-2023
66,WB_FINDEX,WB,Global Findex Database,World Bank (WB),"Prosperity, People, Digital",280,162,2011-2024
124,IMF_GFS_SOO,IMF,Government Finance Statistics (GFS): Statement...,NaN,,218,166,2017-2025
38,BS_SGI,BS,Sustainable Governance Indicators (SGI),Bertelsmann Stiftung Foundation,"Prosperity, Infrastructure",193,41,2014-2024
76,WB_HNP,WB,Health Nutrition and Population Statistics,World Bank (WB),People,192,217,1960-2023


Data360 reúne bases del Banco Mundial y de otras 51 organizaciones. Para agruparlas no conviene usar la columna `fuente`: 22 bases no la informan y algunas organizaciones aparecen escritas de varias formas (la OCDE, de tres). En cambio, el código de cada base siempre empieza con la sigla de quien la produce:

In [5]:
por_organizacion = (
    bases.groupby("sigla", as_index=False)
    .agg(bases=("base", "count"), indicadores=("indicadores", "sum"))
    .nlargest(12, "indicadores")
)
alt.Chart(por_organizacion).mark_bar(color=AZUL, size=16, cornerRadiusEnd=4).encode(
    x=alt.X("indicadores:Q", title="Indicadores"),
    y=alt.Y("sigla:N", title=None, sort="-x"),
    tooltip=[alt.Tooltip("sigla:N", title="Organización"), alt.Tooltip("bases:Q", title="Bases"),
             alt.Tooltip("indicadores:Q", title="Indicadores")],
).properties(width=420, height=alt.Step(24), title="Las 12 organizaciones con más indicadores en Data360")

alt.Chart(...)

`WB` Banco Mundial · `IMF` Fondo Monetario Internacional · `WEF` Foro Económico Mundial · `BS` Fundación Bertelsmann · `OECD` OCDE · `FAO` Organización de las Naciones Unidas para la Alimentación y la Agricultura · `GI` Global Integrity · `VDEM` Varieties of Democracy · `OWID` Our World in Data · `ITU` Unión Internacional de Telecomunicaciones · `FH` Freedom House · `UNDRR` Oficina de la ONU para la Reducción del Riesgo de Desastres.

El Banco Mundial y el FMI concentran más del 80% de los indicadores.

Cada base además está clasificada en uno o más de los cinco grandes temas del Banco Mundial (*WB Practice Groups*):

In [6]:
bases["tema"].replace("", "(sin tema)").str.split(", ").explode().value_counts()

tema
Prosperity        70
(sin tema)        44
Infrastructure    33
Planet            32
People            17
Digital           15
Name: count, dtype: int64

### 1.3 ¿Qué indicadores hay en cada base?
La columna `indicadores` de la tabla anterior ya da la cantidad por base. Para ver cuáles son, `GET /indicators` devuelve los códigos:

In [7]:
codigos_wdi = api_get("indicators", datasetId="WB_WDI")
print(f"World Development Indicators (WB_WDI): {len(codigos_wdi)} indicadores. Primeros códigos: {codigos_wdi[:5]}")

World Development Indicators (WB_WDI): 1499 indicadores. Primeros códigos: ['WB_WDI_AG_LND_EL5M_RU_K2', 'WB_WDI_AG_LND_TOTL_RU_K2', 'WB_WDI_AG_PRD_CROP_XD', 'WB_WDI_AG_PRD_FOOD_XD', 'WB_WDI_AG_PRD_LVSK_XD']


Los códigos solos no dicen mucho. `POST /metadata` acepta filtros en sintaxis OData y devuelve nombre, definición y fuente de cada indicador. Por ejemplo, los indicadores de las Perspectivas de la Economía Mundial del FMI (`IMF_WEO`):

In [8]:
fichas = api_post("metadata", {"query": "&$filter=series_description/database_id eq 'IMF_WEO'&$select=series_description/idno,series_description/name"})
indicadores_weo = pd.DataFrame([ficha["series_description"] for ficha in fichas["value"]])
print(f"IMF_WEO: {len(indicadores_weo)} indicadores")
indicadores_weo.head(10)

IMF_WEO: 45 indicadores


,idno,name
0,IMF_WEO_BCA,"Current account balance (credit less debit), U..."
1,IMF_WEO_GGX_NGDP,"Expenditure, General government, Percent of GDP"
2,IMF_WEO_NGDP_RPCH,"Gross domestic product (GDP), Constant prices,..."
3,IMF_WEO_GGXCNL,"Net lending (+) / net borrowing (-), General g..."
4,IMF_WEO_GGXONLB_NGDP,"Primary net lending (+) / net borrowing (-), G..."
5,IMF_WEO_LUR,Unemployment rate
6,IMF_WEO_NGDPPC,"Gross domestic product (GDP), Current prices, ..."
7,IMF_WEO_TM_RPCH,"Imports of goods and services, Volume, Cost in..."
8,IMF_WEO_GGXONLB,"Primary net lending (+) / net borrowing (-), G..."
9,IMF_WEO_GGXWDG,"Gross debt, General government, Domestic currency"


El buscador también sirve para encontrar un tema en todas las bases a la vez. Buscamos `inflation`:

In [9]:
resultado = api_post("searchv2", {
    "search": "inflation", "filter": "type eq 'indicator'", "top": 10, "count": True,
    "select": "series_description/database_id, series_description/idno, series_description/name",
})
print(f"{resultado['@odata.count']} indicadores relacionados con 'inflation'. Los 10 más relevantes:")
pd.DataFrame([r["series_description"] for r in resultado["value"]])[["database_id", "idno", "name"]]

71 indicadores relacionados con 'inflation'. Los 10 más relevantes:


,database_id,idno,name
0,FAO_CP,FAO_CP_23014,Food price inflation
1,WB_WDI,WB_WDI_FP_CPI_TOTL_ZG,"Inflation, consumer prices (annual % growth)"
2,WB_GS,WB_GS_FP_CPI_TOTL_ZG,"Inflation, consumer prices (annual %)"
3,WB_WDI,WB_WDI_NY_GDP_DEFL_KD_ZG,"Inflation, GDP deflator (annual % growth)"
4,WB_WDI,WB_WDI_NY_GDP_DEFL_KD_ZG_AD,"Inflation, GDP deflator, linked series (annual..."
5,WEF_GCIHH,WEF_GCIHH_INFLAYRAVG,"Inflation, annual % change"
6,WEF_GCI,WEF_GCI_INFLAYRAVG,GCI 4.0: Inflation (%)
7,BS_SGI,BS_SGI_6,Sustainable Governance Indicators: Inflation
8,WB_MPO,FPCPITOTL,Inflation Rate
9,WB_WDI,WB_WDI_FP_WPI_TOTL,Wholesale price index (2010 = 100)


Un mismo concepto aparece en varias bases, con distinta cobertura y metodología. Por eso, antes de graficar hay que elegir la fuente con criterio: cuántos países y años cubre, y si la metodología es la misma para todos.

### 1.4 ¿Cómo se estructuran los datos?
`GET /data` sigue el estándar estadístico **SDMX**: cada fila es una única observación, identificada por la combinación de sus dimensiones. Así se ve una fila (población de Argentina en 2024):

In [10]:
observacion = api_get("data", DATABASE_ID="WB_WDI", INDICATOR="WB_WDI_SP_POP_TOTL", REF_AREA="ARG", timePeriodFrom=2024, timePeriodTo=2024)
pd.Series(observacion["value"][0])

OBS_VALUE                     45696159
TIME_FORMAT                        P1Y
UNIT_MULT                            0
COMMENT_OBS                       None
OBS_STATUS                           A
OBS_CONF                            PU
AGG_METHOD                          _Z
DECIMALS                             2
COMMENT_TS                        None
DATA_SOURCE                       None
LATEST_DATA                      False
SEGMENT                           None
EV_CRITERIA                       None
DATABASE_ID                     WB_WDI
INDICATOR           WB_WDI_SP_POP_TOTL
REF_AREA                           ARG
SEX                                 _T
AGE                                 _T
URBANISATION                        _T
COMP_BREAKDOWN_1                    _Z
COMP_BREAKDOWN_2                    _Z
COMP_BREAKDOWN_3                    _Z
TIME_PERIOD                       2024
FREQ                                 A
UNIT_MEASURE                        PS
UNIT_TYPE                

| Grupo | Campos | Qué indican |
|---|---|---|
| Qué se mide | `DATABASE_ID`, `INDICATOR`, `UNIT_MEASURE` | Base, indicador y unidad (`PS` = personas, `PC_A` = % anual, `USD_K_2015` = dólares constantes de 2015) |
| Dónde | `REF_AREA` | País (código ISO de 3 letras) **o** agregado (`WLD` = mundo, `LCN` = América Latina y el Caribe, etc.) |
| Cuándo | `TIME_PERIOD`, `FREQ` | Período y frecuencia (`A` = anual) |
| Desagregaciones | `SEX`, `AGE`, `URBANISATION`, `COMP_BREAKDOWN_1..3` | `_T` = total, `_Z` = no aplica |
| El valor | `OBS_VALUE` | **Viene como texto**: hay que convertirlo a número |
| Atributos | `OBS_STATUS`, `OBS_CONF`, `UNIT_MULT`, `DECIMALS`… | Estado del dato, confidencialidad, multiplicador, decimales |

En los WDI casi todo es "total", pero otras bases sí desagregan. Por ejemplo, el desempleo de la base de Estadísticas de Género (`WB_GS`). `GET /disaggregation` lista los valores posibles de cada dimensión:

In [11]:
dimensiones = pd.DataFrame(api_get("disaggregation", datasetId="WB_GS", indicatorId="WB_GS_SL_UEM_ZS"))
dimensiones.assign(cantidad=dimensiones["field_value"].str.len(), ejemplos=dimensiones["field_value"].str[:6]).drop(columns="field_value")

,field_name,cantidad,ejemplos
0,SEX,3,"[M, F, _T]"
1,AGE,2,"[Y_GE15, Y15T24]"
2,UNIT_MEASURE,1,[PT]
3,URBANISATION,1,[_T]
4,COMP_BREAKDOWN_1,2,"[GS_TRW, GS_TCZ]"
5,COMP_BREAKDOWN_2,1,[_Z]
6,COMP_BREAKDOWN_3,1,[_Z]
7,FREQ,1,[A]
8,REF_AREA,265,"[BRN, CHL, ECU, GHA, IBD, LTE]"
9,TIME_PERIOD,64,"[2000, 2004, 1988, 1979, 1989, 2018]"


Y la misma tasa de desempleo de Argentina en 2023, por sexo (`F`, `M`, `_T`) y por edad (`Y15T24` = 15 a 24 años, `Y_GE15` = 15 años o más):

In [12]:
desempleo = obtener_datos("WB_GS", "WB_GS_SL_UEM_ZS", REF_AREA="ARG", timePeriodFrom=2023, timePeriodTo=2023)
desempleo.assign(valor=pd.to_numeric(desempleo["OBS_VALUE"])).pivot(index="AGE", columns="SEX", values="valor")

SEX,F,M,_T
AGE,,,
Y15T24,20.063,17.944,18.822
Y_GE15,7.583,6.287,6.841


La ficha de cada indicador (`POST /metadata`) explica qué mide. Por ejemplo, la inflación del WDI:

In [13]:
ficha = api_post("metadata", {"query": "&$filter=series_description/idno eq 'WB_WDI_FP_CPI_TOTL_ZG'"})["value"][0]["series_description"]
pd.Series({campo: ficha[campo] for campo in ["name", "definition_long", "measurement_unit", "periodicity", "aggregation_method"]})

name                       Inflation, consumer prices (annual % growth)
definition_long       Inflation as measured by the consumer price in...
measurement_unit                                                      %
periodicity                                                      Annual
aggregation_method                                               Median
dtype: str

Por último, la geografía. `REF_AREA` mezcla países con agregados (regiones, grupos de ingreso). La jerarquía del portal dice qué códigos son países y a qué región pertenece cada uno:

In [14]:
jerarquia = api_get("portal/v1/hierarchy", type="country")["data"]
paises = {pais["id"]: pais["name"] for region in jerarquia["HCL_WB_REGIONS"] for pais in region["countries"]}
print(f"{len(paises)} países en {len(jerarquia['HCL_WB_REGIONS'])} regiones")
pd.DataFrame([(region["name"], len(region["countries"])) for region in jerarquia["HCL_WB_REGIONS"]], columns=["Región", "Países"])

220 países en 8 regiones


,Región,Países
0,Eastern and Southern Africa,26
1,Western and Central Africa,22
2,East Asia & Pacific,38
3,Europe & Central Asia,58
4,Latin America & Caribbean,44
5,North America,3
6,"Middle East, North Africa, Afghanistan & Pakistan",23
7,South Asia,6


### 1.5 Lo que aprendimos de la API
- **Catálogo:** 174 bases de 52 organizaciones, con casi 10.000 indicadores. El Banco Mundial y el FMI concentran más del 80%.
- **Temas:** las bases se clasifican en cinco temas (Prosperidad, Infraestructura, Planeta, Personas y Digital), aunque 44 no tienen tema asignado.
- **Estructura:** formato SDMX, con una fila por indicador × país × período × desagregación. Los códigos `_T` y `_Z` marcan totales y dimensiones que no aplican. `REF_AREA` mezcla países con agregados regionales, y la jerarquía del portal permite separarlos.
- **Paginación:** `/data` devuelve como máximo 1000 filas por llamada, y el buscador también. Hay que paginar con `skip`.
- **Filtros:** `REF_AREA` acepta varios países separados por coma. El filtro de período **solo funciona si se pasan `timePeriodFrom` y `timePeriodTo` juntos**: con uno solo, la API devuelve la serie completa.
- **Detalles a cuidar:** `OBS_VALUE` llega como texto. El nombre de la organización en las fichas falta o es inconsistente. Los conteos no siempre coinciden: `/indicators` lista 1499 códigos para el WDI y la ficha de la base dice 1545.

## 2. Visualizaciones (punto 2)
**Tema: la excepción argentina.** En los años 80 casi toda América Latina convivía con inflación alta. Hoy casi todos los países la bajaron a un dígito. Argentina no, y su economía dejó de crecer. Lo contamos con tres preguntas, una por gráfico:

1. ¿Cómo evolucionó la inflación en la región? → **mapa de calor** país × año
2. ¿Qué pasó con el ingreso por habitante? → **líneas** del PIB per cápita
3. ¿Van de la mano las dos cosas? → **dispersión** inflación vs. crecimiento

En los tres gráficos, Argentina va en azul y el resto de los países en gris.

### 2.1 Los datos
Tres indicadores del WDI para todos los países del mundo, de 1960 a 2025. El filtro `REF_AREA` usa la lista de países de la sección 1.4, así quedan afuera los agregados regionales:

In [15]:
INDICADORES = {
    "ipc": "WB_WDI_FP_CPI_TOTL_ZG",           # Inflación, precios al consumidor (% anual)
    "deflactor": "WB_WDI_NY_GDP_DEFL_KD_ZG",  # Inflación, deflactor del PIB (% anual)
    "pib_pc": "WB_WDI_NY_GDP_PCAP_KD",        # PIB per cápita (US$ constantes de 2015)
}
LATAM = {
    "ARG": "Argentina", "BOL": "Bolivia", "BRA": "Brasil", "CHL": "Chile", "COL": "Colombia", "CRI": "Costa Rica",
    "DOM": "Rep. Dominicana", "ECU": "Ecuador", "SLV": "El Salvador", "GTM": "Guatemala", "HND": "Honduras", "MEX": "México",
    "NIC": "Nicaragua", "PAN": "Panamá", "PRY": "Paraguay", "PER": "Perú", "URY": "Uruguay", "VEN": "Venezuela",
}
filtros = {"REF_AREA": ",".join(paises), "timePeriodFrom": 1960, "timePeriodTo": 2025}

wdi = (
    pd.concat([obtener_datos("WB_WDI", codigo, **filtros).assign(variable=variable) for variable, codigo in INDICADORES.items()])
    .assign(anio=lambda d: d["TIME_PERIOD"].astype(int), valor=lambda d: pd.to_numeric(d["OBS_VALUE"]))
    .pivot(index=["REF_AREA", "anio"], columns="variable", values="valor")  # una fila por país y año, una columna por indicador
    .reset_index()
    .rename_axis(columns=None)
)
wdi.sample(5, random_state=1)

,REF_AREA,anio,deflactor,ipc,pib_pc
8357,PAN,1962,0.571619,0.788002,2876.856903
7777,NGA,1966,13.192871,9.690346,1528.594504
988,BHR,1969,NaN,-0.194128,NaN
5067,IRN,2012,18.953884,27.256813,5058.646378
4412,GUY,1964,-0.485363,NaN,2587.337457


**¿Qué medida de inflación usar?** La habitual es el índice de precios al consumidor (IPC). Pero al contar los años con dato aparece un problema: justo Argentina y Venezuela, los dos países con más inflación de la región, casi no tienen IPC en el WDI.
- **Argentina:** la serie recién empieza en 2018. Es el primer año que se puede calcular con el índice que el INDEC publica desde fines de 2016, después de la intervención de 2007-2015. La serie anterior no se usa.
- **Venezuela:** solo hay datos de 2009 a 2016.

El deflactor del PIB, que mide la inflación del conjunto de la economía, sí tiene las series completas:

In [16]:
diferencia = (wdi["ipc"] - wdi["deflactor"]).abs().median()
print(f"En los años con las dos medidas, la diferencia mediana entre IPC y deflactor es de {diferencia:.1f} puntos porcentuales")
wdi[wdi["REF_AREA"].isin(LATAM)].groupby("REF_AREA")[["ipc", "deflactor"]].count().rename(index=LATAM).sort_values("ipc").head(5)

En los años con las dos medidas, la diferencia mediana entre IPC y deflactor es de 1.9 puntos porcentuales


,ipc,deflactor
REF_AREA,,
Argentina,7,65
Venezuela,8,65
Nicaragua,26,65
Brasil,45,65
Chile,55,65


El deflactor completa los huecos y en general se parece al IPC. Pero conviene revisarlo, porque también se usa para pasar el PIB a dólares constantes. Buscamos saltos del PIB per cápita de más de 15% en un año, que son muy raros:

In [17]:
variacion_pib = 100 * (wdi["pib_pc"] / wdi.groupby("REF_AREA")["pib_pc"].shift() - 1)  # respecto del año anterior
saltos = wdi[wdi["REF_AREA"].isin(LATAM) & (variacion_pib.abs() > 15)]
saltos.assign(pais=saltos["REF_AREA"].map(LATAM), variacion=variacion_pib.round(1))[["pais", "anio", "variacion"]]

,pais,anio,variacion
2949,Rep. Dominicana,1965,-15.0
4569,Honduras,1990,17.8
4570,Honduras,1991,22.5
4573,Honduras,1994,25.4
4574,Honduras,1995,21.2
4575,Honduras,1996,19.5
4576,Honduras,1997,19.1
7856,Nicaragua,1979,-28.4
8383,Panamá,1988,-15.3
8415,Panamá,2020,-18.9


Casi todos los saltos tienen una explicación histórica: la guerra civil dominicana (1965), la revolución nicaragüense (1979), la guerra civil salvadoreña (1980), la crisis panameña de 1988, la pandemia (2020-2021) y los vaivenes de Venezuela (la crisis de 1989, el rebote de 2004 tras el paro petrolero y el colapso desde 2016).

Los de Honduras en los 90, no: su PIB per cápita casi se cuadruplica en ocho años. El problema es el deflactor, que en esos años no pasa de 6% anual mientras el IPC llega a 34%. Al descontar menos inflación de la que hubo, el PIB "real" crece de más:

In [18]:
wdi[(wdi["REF_AREA"] == "HND") & wdi["anio"].between(1989, 1998)].set_index("anio")[["ipc", "deflactor", "pib_pc"]].round(1).T

anio,1989,1990,1991,1992,1993,1994,1995,1996,1997,1998
ipc,9.9,23.3,34.0,8.8,10.7,21.7,29.5,23.8,20.2,13.7
deflactor,3.9,-0.1,1.6,5.7,6.0,-2.0,4.1,3.1,5.3,2.1
pib_pc,388.8,458.0,561.1,594.7,656.9,823.8,998.4,1193.4,1420.7,1543.8


**Decisiones:**
1. La inflación es el IPC y, solo donde falta, el deflactor del PIB. La columna `medida` registra cuál se usó en cada caso.
2. Descartamos el PIB de Honduras anterior a 2000.

In [19]:
wdi["inflacion"] = wdi["ipc"].fillna(wdi["deflactor"])
wdi["medida"] = wdi["ipc"].notna().map({True: "IPC", False: "Deflactor del PIB"})
wdi.loc[(wdi["REF_AREA"] == "HND") & (wdi["anio"] < 2000), "pib_pc"] = None

latam = wdi[wdi["REF_AREA"].isin(LATAM)].assign(pais=lambda d: d["REF_AREA"].map(LATAM))
es_argentina = alt.datum.pais == "Argentina"  # condición que usan los tres gráficos para destacar a Argentina

### 2.2 Gráfico 1: medio siglo de inflación en América Latina
Un **mapa de calor**: cada fila es un país, cada columna un año, y el color indica en qué rango cayó la inflación. Los rangos vuelven legible una variable que va de 0% a más de 10.000%. Los países se ordenan por su inflación mediana desde 2011, así que los casos actuales quedan arriba.

*Interacción:* al pasar el mouse se resalta la fila del país, y el tooltip muestra el valor exacto y la medida usada.

In [20]:
RANGOS = ["< 10%", "10–25%", "25–50%", "50–100%", "100–1.000%", "> 1.000%"]
RAMPA_INFLACION = ["#fee1d6", "#f7a284", "#e87043", "#c64a0f", "#953404", "#632308"]  # un solo tono, de claro a oscuro

datos_inflacion = latam.dropna(subset=["inflacion"]).assign(
    rango=lambda d: pd.cut(d["inflacion"], [-float("inf"), 10, 25, 50, 100, 1000, float("inf")], labels=RANGOS, right=False)
)[["pais", "anio", "inflacion", "medida", "rango"]].round(1)
orden_paises = datos_inflacion[datos_inflacion["anio"] >= 2011].groupby("pais")["inflacion"].median().sort_values(ascending=False).index.tolist()

fila_resaltada = alt.selection_point(on="pointerover", clear="pointerout", fields=["pais"])

fig_inflacion = (
    alt.Chart(datos_inflacion)
    .mark_rect(stroke=FONDO, strokeWidth=1)
    .encode(
        x=alt.X("anio:O", title=None, axis=alt.Axis(values=list(range(1970, 2030, 10)), labelAngle=0, ticks=False, domain=False)),
        y=alt.Y("pais:N", title=None, sort=orden_paises, axis=alt.Axis(
            ticks=False, domain=False, labelPadding=6,
            labelColor=alt.expr(f"datum.value == 'Argentina' ? '{TINTA}' : '{TINTA_SECUNDARIA}'"),
            labelFontWeight=alt.expr("datum.value == 'Argentina' ? 700 : 400"),
        )),
        color=alt.Color("rango:O", title=None, scale=alt.Scale(domain=RANGOS, range=RAMPA_INFLACION),
                        legend=alt.Legend(orient="top", direction="horizontal", columns=3, symbolType="square", columnPadding=12)),
        opacity=alt.when(fila_resaltada).then(alt.value(1)).otherwise(alt.value(0.3)),
        tooltip=[alt.Tooltip("pais:N", title="País"), alt.Tooltip("anio:O", title="Año"),
                 alt.Tooltip("inflacion:Q", title="Inflación (%)", format=",.1f"), alt.Tooltip("medida:N", title="Medida")],
    )
    .add_params(fila_resaltada)
    .properties(
        width=720, height=400,
        title=alt.Title("América Latina venció a la inflación. Argentina, no",
                        subtitle=["Inflación anual por país (IPC; donde falta, deflactor del PIB).",
                                  "Las celdas vacías son años sin datos."]),
    )
)
fig_inflacion

alt.Chart(...)

### 2.3 Gráfico 2: el ingreso por habitante
Un gráfico de **líneas con énfasis**: Argentina en azul, un país de comparación en gris oscuro y el resto de la región de fondo. El PIB per cápita está en dólares constantes de 2015, es decir, sin el efecto de la inflación.

*Interacción:* un menú permite elegir el país de comparación. Al mover el mouse, una línea vertical marca el año más cercano y un tooltip muestra el valor de los dos países.

In [21]:
datos_pib = latam.dropna(subset=["pib_pc"])[["pais", "anio", "pib_pc"]].round(1)

pais_comparado = alt.param(name="comparar", value="Chile", bind=alt.binding_select(
    options=sorted(pais for pais in LATAM.values() if pais != "Argentina"), name="Comparar con "))
anio_cercano = alt.selection_point(fields=["anio"], nearest=True, on="pointerover", clear="pointerout", empty=False)
color_destacado = alt.when(es_argentina).then(alt.value(AZUL)).otherwise(alt.value(TINTA_SECUNDARIA))
eje_x = alt.X("anio:Q", title=None, scale=alt.Scale(domain=[1960, 2025]), axis=alt.Axis(format="d", grid=False))

es_destacado = es_argentina | (alt.datum.pais == pais_comparado)  # Argentina y el país elegido

base = alt.Chart(datos_pib).encode(x=eje_x, y=alt.Y("pib_pc:Q", title=None, axis=alt.Axis(format="$,.0f", domain=False, ticks=False, tickCount=5)))
destacados = base.transform_filter(es_destacado)
por_anio = (  # una fila por año con los dos valores lado a lado, para la línea vertical y el tooltip
    alt.Chart(datos_pib)
    .transform_filter(es_destacado)
    .transform_calculate(serie="datum.pais == 'Argentina' ? 'argentina' : 'elegido'")
    .transform_pivot("serie", value="pib_pc", groupby=["anio"])
    .encode(x=eje_x)
)

fig_pib = (
    alt.layer(
        base.mark_line(color=EJE, strokeWidth=1).encode(detail="pais:N"),                       # resto de la región, de fondo
        destacados.mark_line(strokeWidth=2.5).encode(color=color_destacado, detail="pais:N"),
        destacados.mark_text(align="left", dx=6, fontWeight=600, color=TINTA).encode(text="pais:N")
        .transform_filter(alt.datum.anio == 2025),                                             # nombres al final de las líneas
        por_anio.mark_rule(color=TINTA_TENUE).transform_filter(anio_cercano),                  # línea vertical en el año elegido
        destacados.mark_circle(size=64, stroke=FONDO, strokeWidth=2).encode(
            color=color_destacado, opacity=alt.when(anio_cercano).then(alt.value(1)).otherwise(alt.value(0))),
        por_anio.mark_point(opacity=0).encode(tooltip=[                                         # capta el mouse: año más cercano
            alt.Tooltip("anio:Q", title="Año", format="d"),
            alt.Tooltip("argentina:Q", title="Argentina", format="$,.0f"),
            alt.Tooltip("elegido:Q", title="País elegido", format="$,.0f"),
        ]).add_params(anio_cercano),
    )
    .add_params(pais_comparado)
    .properties(
        width=720, height=380,
        title=alt.Title("En 1960 Argentina era el país más rico de la región. Hoy es el quinto",
                        subtitle=["PIB por habitante, en dólares constantes de 2015. Elegí un país para compararlo;",
                                  "en gris, el resto de América Latina."]),
    )
)
fig_pib

alt.LayerChart(...)

### 2.4 Gráfico 3: inflación y crecimiento, juntos
Un **gráfico de dispersión** que cruza las dos variables anteriores para cada país, entre 2011 (el año del pico del PIB per cápita argentino) y 2025:
- **Eje x:** la inflación anual mediana, en escala logarítmica para que entren desde el menos de 1% de Panamá hasta el más de 200% de Venezuela.
- **Eje y:** cuánto creció o cayó el PIB per cápita en esos 14 años.

*Interacción:* cada punto tiene un área de hover de unos 28 px, más grande que el punto. Al pasar el mouse, el punto se agranda y un tooltip muestra sus valores.

In [22]:
periodo = latam[latam["anio"].between(2011, 2025)]
pib_extremos = periodo.pivot(index="pais", columns="anio", values="pib_pc")
datos_dispersion = pd.DataFrame({
    "inflacion_mediana": periodo.groupby("pais")["inflacion"].median(),
    "variacion_pib": 100 * (pib_extremos[2025] / pib_extremos[2011] - 1),
}).reset_index().round(1)

ETIQUETADOS = ["Argentina", "Venezuela", "Rep. Dominicana", "Costa Rica", "Uruguay", "Brasil"]  # los que no se pisan
a_la_izquierda = "datum.pais == 'Venezuela'"  # está en el borde derecho: su nombre va a la izquierda del punto
pais_resaltado = alt.selection_point(on="pointerover", clear="pointerout", fields=["pais"], empty=False)
porcentaje = "datum.label + '%'"

puntos = alt.Chart(datos_dispersion).encode(
    x=alt.X("inflacion_mediana:Q", title="Inflación anual mediana 2011-2025 (escala logarítmica)",
            scale=alt.Scale(type="log", domain=[0.5, 400]), axis=alt.Axis(values=[0.5, 1, 2, 5, 10, 20, 50, 100, 200], labelExpr=porcentaje)),
    y=alt.Y("variacion_pib:Q", title="Variación del PIB per cápita 2011-2025", axis=alt.Axis(labelExpr=porcentaje, domain=False, ticks=False)),
)

fig_dispersion = (
    alt.layer(
        alt.Chart().mark_rule(color=EJE).encode(y=alt.datum(0)),                      # 0%: mismo PIB per cápita que en 2011
        alt.Chart().mark_rule(color=EJE).encode(x=alt.datum(10)),                     # 10%: fin de la inflación de un dígito
        alt.Chart().mark_text(text="Inflación de dos dígitos →", align="left", baseline="top", dx=6, color=TINTA_TENUE)
        .encode(x=alt.datum(10), y=alt.value(0)),
        alt.Chart().mark_text(text="↓ Cayó desde 2011", align="left", baseline="top", dx=4, dy=6, color=TINTA_TENUE)
        .encode(x=alt.value(0), y=alt.datum(0)),                                    # x en píxeles: borde izquierdo
        puntos.mark_circle(opacity=1, stroke=FONDO, strokeWidth=2).encode(
            color=alt.when(es_argentina).then(alt.value(AZUL)).otherwise(alt.value(TINTA_TENUE)),
            size=alt.when(pais_resaltado).then(alt.value(260)).otherwise(alt.value(110)),
        ),
        puntos.mark_text(
            align=alt.expr(f"{a_la_izquierda} ? 'right' : 'left'"), dx=alt.expr(f"{a_la_izquierda} ? -9 : 9"),
            fontWeight=alt.expr("datum.pais == 'Argentina' ? 700 : 400"),
        ).encode(
            text="pais:N", color=alt.when(es_argentina).then(alt.value(TINTA)).otherwise(alt.value(TINTA_SECUNDARIA)),
        ).transform_filter(alt.FieldOneOfPredicate(field="pais", oneOf=ETIQUETADOS)),  # nombres de algunos países
        puntos.mark_circle(size=600, opacity=0).encode(tooltip=[                       # área de hover invisible (~28 px)
            alt.Tooltip("pais:N", title="País"),
            alt.Tooltip("inflacion_mediana:Q", title="Inflación mediana (%)", format=",.1f"),
            alt.Tooltip("variacion_pib:Q", title="Variación del PIB per cápita (%)", format="+,.1f"),
        ]).add_params(pais_resaltado),
    )
    .properties(
        width=720, height=420,
        title=alt.Title("Inflación alta, economías en retroceso",
                        subtitle=["Cada punto es un país de América Latina. Solo Argentina y Venezuela tuvieron una inflación",
                                  "mediana de dos dígitos o más, y son los únicos que hoy producen menos por habitante que en 2011."]),
    )
)
fig_dispersion

alt.LayerChart(...)

### 2.5 Exportación para la web
Cada gráfico se guarda en dos archivos:
- **JSON de Vega-Lite** (`docs/graficos/`): la landing lo dibuja con `vega-embed`. El ancho se adapta a la pantalla (`width="container"`) y el alto queda fijo.
- **CSV con sus datos** (`docs/datos/`): para que el lector los pueda descargar.

In [23]:
SALIDA = Path("docs")
(SALIDA / "graficos").mkdir(parents=True, exist_ok=True)
(SALIDA / "datos").mkdir(parents=True, exist_ok=True)


def exportar(nombre, grafico, datos):
    grafico.properties(width="container").save(SALIDA / "graficos" / f"{nombre}.json")
    datos.to_csv(SALIDA / "datos" / f"{nombre}.csv", index=False)


exportar("inflacion", fig_inflacion, datos_inflacion)
exportar("pib", fig_pib, datos_pib)
exportar("inflacion_vs_pib", fig_dispersion, datos_dispersion)
sorted(archivo.as_posix() for archivo in SALIDA.glob("*/*"))

['docs/datos/inflacion.csv',
 'docs/datos/inflacion_vs_pib.csv',
 'docs/datos/pib.csv',
 'docs/graficos/inflacion.json',
 'docs/graficos/inflacion_vs_pib.json',
 'docs/graficos/pib.json']

## 3. Storytelling (punto 3)
La historia se publica como landing en GitHub Pages ([`docs/index.html`](docs/index.html)), con los tres gráficos exportados en la sección 2.5. Estos son los números que cita la nota, calculados con los mismos datos.

**La inflación**

In [24]:
argentina = latam[latam["pais"] == "Argentina"].set_index("anio")
un_digito = argentina.loc[1961:2025, "inflacion"] < 10
print(f"Argentina tuvo inflación de un dígito en {un_digito.sum()} de {un_digito.size} años (1961-2025): {un_digito[un_digito].index.tolist()}")
for anio in [2024, 2025]:
    print(f"Inflación argentina en {anio}: {argentina.loc[anio, 'inflacion']:.1f}% ({argentina.loc[anio, 'medida']})")

inflacion_por_anio = latam.pivot(index="anio", columns="pais", values="inflacion")
for anio in [1990, 2025]:
    fila = inflacion_por_anio.loc[anio]
    print(f"{anio}: {(fila < 10).sum()} de {fila.notna().sum()} países con inflación de un dígito, mediana regional de "
          f"{fila.median():.1f}%. Con más de 100%: {fila[fila > 100].index.tolist()}")

Argentina tuvo inflación de un dígito en 11 de 65 años (1961-2025): [1969, 1970, 1993, 1994, 1995, 1996, 1997, 1998, 1999, 2000, 2001]
Inflación argentina en 2024: 219.9% (IPC)
Inflación argentina en 2025: 39.1% (Deflactor del PIB)
1990: 1 de 18 países con inflación de un dígito, mediana regional de 39.2%. Con más de 100%: ['Argentina', 'Brasil', 'Nicaragua', 'Perú', 'Uruguay']
2025: 15 de 18 países con inflación de un dígito, mediana regional de 4.0%. Con más de 100%: ['Venezuela']


**El ingreso por habitante**

In [25]:
pib_por_anio = latam.pivot(index="anio", columns="pais", values="pib_pc")
caida = 100 * (1 - pib_por_anio.loc[2025, "Argentina"] / pib_por_anio.loc[2011, "Argentina"])
print(f"PIB per cápita argentino: máximo en {pib_por_anio['Argentina'].idxmax()} (US$ {pib_por_anio.loc[2011, 'Argentina']:.0f}). "
      f"En 2025, US$ {pib_por_anio.loc[2025, 'Argentina']:.0f}: un {caida:.1f}% menos")
print(f"En 1960 era {pib_por_anio.loc[1960, 'Argentina'] / pib_por_anio.loc[1960, 'Chile']:.1f} veces el de Chile")

puesto = pib_por_anio.loc[[1960, 2025]].rank(axis=1, ascending=False)["Argentina"]
print(f"Puesto de Argentina en la región: {puesto[1960]:.0f}° en 1960 y {puesto[2025]:.0f}° en 2025")

superan = pib_por_anio.gt(pib_por_anio["Argentina"], axis=0)  # True si el país supera a Argentina ese año
print("Países que hoy superan a Argentina, y desde qué año lo hacen sin interrupción:")
for pais in superan.columns[superan.loc[2025]]:
    print(f"  {pais}: desde {superan.index[~superan[pais]].max() + 1}")

PIB per cápita argentino: máximo en 2011 (US$ 14041). En 2025, US$ 13287: un 5.4% menos
En 1960 era 2.3 veces el de Chile
Puesto de Argentina en la región: 1° en 1960 y 5° en 2025
Países que hoy superan a Argentina, y desde qué año lo hacen sin interrupción:
  Chile: desde 2018
  Costa Rica: desde 2019
  Panamá: desde 2014
  Uruguay: desde 2009


**En el mundo:** ¿cuántos países tuvieron una inflación mediana de 20% o más entre 2011 y 2025? Contamos solo los que tienen dato en al menos 12 de esos 15 años.

In [26]:
mundo = wdi[wdi["anio"].between(2011, 2025)].groupby("REF_AREA")["inflacion"].agg(["median", "count"])
club = mundo[(mundo["median"] >= 20) & (mundo["count"] >= 12)].sort_values("median", ascending=False)
club.assign(pais=club.index.map(paises)).round(1)

,median,count,pais
REF_AREA,,,
VEN,227.1,15,"Venezuela, RB"
SSD,46.2,14,South Sudan
ARG,40.3,15,Argentina
SYR,37.6,12,Syrian Arab Republic
SDN,36.9,15,Sudan
IRN,30.6,15,"Iran, Islamic Rep."
MWI,21.3,15,Malawi


**Datos para Flourish y Datawrapper.** La nota suma dos gráficos hechos con herramientas vistas en la materia:
- Un ranking animado del PIB per cápita, en Flourish.
- Un mapa de la inflación de 2025, en Datawrapper.

Acá preparamos sus datos en el formato que pide cada herramienta. Los pasos para armarlos están en `guia_flourish_datawrapper.md`.

In [27]:
# Flourish ("Bar chart race"): una fila por país y una columna por año
ranking_pib = pib_por_anio.T.round().astype("Int64")
ranking_pib.insert(0, "grupo", ["Argentina" if pais == "Argentina" else "Resto de América Latina" for pais in ranking_pib.index])
ranking_pib.to_csv(SALIDA / "datos" / "flourish_ranking_pib.csv", index_label="pais")

# Datawrapper (mapa "Latin America"): el código ISO de 3 letras ubica a cada país en el mapa
mapa_inflacion = latam.loc[latam["anio"] == 2025, ["REF_AREA", "pais", "inflacion", "medida"]].rename(columns={"REF_AREA": "iso3"})
mapa_inflacion.round(1).to_csv(SALIDA / "datos" / "datawrapper_mapa_inflacion.csv", index=False)
ranking_pib.iloc[:5, :6]

anio,grupo,1960,1961,1962,1963,1964
pais,,,,,,
Argentina,Argentina,7397,7671,7480,6968,7551
Bolivia,Resto de América Latina,1277,1277,1321,1382,1406
Brasil,Resto de América Latina,2604,2743,2838,2773,2785
Chile,Resto de América Latina,3223,3331,3391,3513,3529
Colombia,Resto de América Latina,1961,1998,2043,2046,2108
